In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

**Load Datasets**

In [2]:
lagged_data = pd.read_csv("LIHEAP_lags.csv")
lagged_data.head(5)

,Zip Code,Month_Year,Enrollment_Count,LIHEAP_lag1,LIHEAP_lag2,LIHEAP_lag3,lag1_status,lag2_status,lag3_status,Complete_Lags,Edge
0,91901,2023-03,2.0,1.0,NaN,NaN,present,zip_gap,before_start,False,start
1,91901,2023-04,1.0,2.0,1.0,NaN,present,present,zip_gap,False,core
2,91901,2023-05,2.0,1.0,2.0,1.0,present,present,present,True,core
3,91901,2023-06,2.0,2.0,1.0,2.0,present,present,present,True,core
4,91901,2023-07,1.0,2.0,2.0,1.0,present,present,present,True,core


In [3]:
policy_annotation = pd.read_csv("Policy_Annotations.csv")
policy_annotation = policy_annotation.rename(columns={"Month Year": "Month_Year"})
policy_annotation.sort_values("Month_Year").reset_index(drop=True)
policy_annotation.head(5)

,Month_Year,CalFresh_Policy_Active,CalFresh_Change_Types,CalFresh_Expected_Direction,CalFresh_Policies,LIHEAP_Policy_Active,LIHEAP_Change_Types,LIHEAP_Expected_Direction,LIHEAP_Policies,FoodBank_Policy_Active,FoodBank_Change_Types,FoodBank_Expected_Direction,FoodBank_Policies,CalFresh_Data_Available,LIHEAP_Data_Available,FoodBank_Data_Available,Any_Policy_Active
0,2018-01,No,NaN,NaN,NaN,No,NaN,NaN,NaN,No,NaN,NaN,NaN,No,No,Yes,No
1,2018-02,No,NaN,NaN,NaN,No,NaN,NaN,NaN,No,NaN,NaN,NaN,No,No,Yes,No
2,2018-03,No,NaN,NaN,NaN,No,NaN,NaN,NaN,No,NaN,NaN,NaN,No,No,Yes,No
3,2018-04,No,NaN,NaN,NaN,No,NaN,NaN,NaN,No,NaN,NaN,NaN,No,No,Yes,No
4,2018-05,No,NaN,NaN,NaN,No,NaN,NaN,NaN,No,NaN,NaN,NaN,No,No,Yes,No


**Define the Change Month FLags computed on the full policy timeline so that the first lag month compares it its true prior month**

In [4]:
progs = ["CalFresh", "LIHEAP", "FoodBank"]

for p in progs:
    s = policy_annotation[f'{p}_Policies'].fillna('')
    policy_annotation[f'{p}_Policy_Change'] = (s != s.shift(1)).astype(int)
    policy_annotation.loc[0, f'{p}_Policy_Change'] = 0
policy_annotation["Any_Policy_Change"] = policy_annotation[[f'{p}_Policy_Change' for p in progs]].max(axis=1) 

**Indicate Policy CHange Periods (period is in effect from the start month onward while active)**

In [5]:
policy_annotation

,Month_Year,CalFresh_Policy_Active,CalFresh_Change_Types,CalFresh_Expected_Direction,CalFresh_Policies,LIHEAP_Policy_Active,LIHEAP_Change_Types,LIHEAP_Expected_Direction,LIHEAP_Policies,FoodBank_Policy_Active,...,FoodBank_Expected_Direction,FoodBank_Policies,CalFresh_Data_Available,LIHEAP_Data_Available,FoodBank_Data_Available,Any_Policy_Active,CalFresh_Policy_Change,LIHEAP_Policy_Change,FoodBank_Policy_Change,Any_Policy_Change
0,2018-01,No,NaN,NaN,NaN,No,NaN,NaN,NaN,No,...,NaN,NaN,No,No,Yes,No,0,0,0,0
1,2018-02,No,NaN,NaN,NaN,No,NaN,NaN,NaN,No,...,NaN,NaN,No,No,Yes,No,0,0,0,0
2,2018-03,No,NaN,NaN,NaN,No,NaN,NaN,NaN,No,...,NaN,NaN,No,No,Yes,No,0,0,0,0
3,2018-04,No,NaN,NaN,NaN,No,NaN,NaN,NaN,No,...,NaN,NaN,No,No,Yes,No,0,0,0,0
4,2018-05,No,NaN,NaN,NaN,No,NaN,NaN,NaN,No,...,NaN,NaN,No,No,Yes,No,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,2025-04,No,NaN,NaN,NaN,Yes,funding,decrease,Federal LIHEAP staff dismissed; FY2025 funds held,No,...,NaN,NaN,Yes,No,Yes,Yes,0,1,0,1
88,2025-05,No,NaN,NaN,NaN,Yes,funding,decrease,Federal LIHEAP staff dismissed; FY2025 funds held,No,...,NaN,NaN,Yes,No,Yes,Yes,0,0,0,0
89,2025-06,No,NaN,NaN,NaN,No,NaN,NaN,NaN,No,...,NaN,NaN,Yes,No,Yes,No,0,1,0,1
90,2025-07,Yes,eligibility,decrease,One Big Beautiful Bill Act,No,NaN,NaN,NaN,Yes,...,increase,One Big Beautiful Bill Act,Yes,No,Yes,Yes,1,0,1,1


In [6]:
policy_annotation["Policy_Period_EA_Ended"] = (policy_annotation["Month_Year"] >= '2023-04').astype(int)
policy_annotation["Policy_Period_ABWD_Resumed"] = (policy_annotation["Month_Year"] >= '2023-07').astype(int)

policy_annotation["Policy_Period_LIHEAP_FY24_Cut"] = policy_annotation["LIHEAP_Policies"].fillna('').str.contains('FY2024 LIHEAP funding cut').astype(int)
policy_annotation["LIHEAP_Direction_Increased"] = (policy_annotation["LIHEAP_Expected_Direction"]=='increase').astype(int)
policy_annotation["LIHEAP_Direction_Decreased"] = (policy_annotation["LIHEAP_Expected_Direction"]=="decrease").astype(int)

In [7]:
merged = lagged_data.merge(policy_annotation, on="Month_Year", how="left", validate="many_to_one", indicator=True)
assert (merged["_merge"]=="both").all(), merged.loc[merged._merge != "both", "Month_Year"].unique()
merged = merged.drop(columns="_merge")
assert len(merged) == len(lagged_data)

In [8]:
merged

,Zip Code,Month_Year,Enrollment_Count,LIHEAP_lag1,LIHEAP_lag2,LIHEAP_lag3,lag1_status,lag2_status,lag3_status,Complete_Lags,...,Any_Policy_Active,CalFresh_Policy_Change,LIHEAP_Policy_Change,FoodBank_Policy_Change,Any_Policy_Change,Policy_Period_EA_Ended,Policy_Period_ABWD_Resumed,Policy_Period_LIHEAP_FY24_Cut,LIHEAP_Direction_Increased,LIHEAP_Direction_Decreased
0,91901,2023-03,2.0,1.0,NaN,NaN,present,zip_gap,before_start,False,...,Yes,0,0,0,0,0,0,0,1,0
1,91901,2023-04,1.0,2.0,1.0,NaN,present,present,zip_gap,False,...,Yes,1,0,1,1,1,0,0,1,0
2,91901,2023-05,2.0,1.0,2.0,1.0,present,present,present,True,...,Yes,0,0,0,0,1,0,0,1,0
3,91901,2023-06,2.0,2.0,1.0,2.0,present,present,present,True,...,Yes,0,0,0,0,1,0,0,1,0
4,91901,2023-07,1.0,2.0,2.0,1.0,present,present,present,True,...,Yes,1,0,1,1,1,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
847,92198,2023-03,NaN,NaN,1.0,NaN,zip_gap,present,before_start,False,...,Yes,0,0,0,0,0,0,0,1,0
848,92198,2023-04,1.0,NaN,NaN,1.0,zip_gap,zip_gap,present,False,...,Yes,1,0,1,1,1,0,0,1,0
849,92198,2023-05,NaN,1.0,NaN,NaN,present,zip_gap,zip_gap,False,...,Yes,0,0,0,0,1,0,0,1,0
850,92198,2023-06,NaN,NaN,1.0,NaN,zip_gap,present,zip_gap,False,...,Yes,0,0,0,0,1,0,0,1,0


In [9]:
merged.dtypes

Zip Code                           int64
Month_Year                           str
Enrollment_Count                 float64
LIHEAP_lag1                      float64
LIHEAP_lag2                      float64
LIHEAP_lag3                      float64
lag1_status                          str
lag2_status                          str
lag3_status                          str
Complete_Lags                       bool
Edge                                 str
CalFresh_Policy_Active               str
CalFresh_Change_Types                str
CalFresh_Expected_Direction          str
CalFresh_Policies                    str
LIHEAP_Policy_Active                 str
LIHEAP_Change_Types                  str
LIHEAP_Expected_Direction            str
LIHEAP_Policies                      str
FoodBank_Policy_Active               str
FoodBank_Change_Types                str
FoodBank_Expected_Direction          str
FoodBank_Policies                    str
CalFresh_Data_Available              str
LIHEAP_Data_Avai

**Perform Data Verification**

In [10]:
flags = ['LIHEAP_Policy_Active','LIHEAP_Expected_Direction','LIHEAP_Policy_Change','CalFresh_Policy_Change','FoodBank_Policy_Change',
         'Policy_Period_EA_Ended','Policy_Period_ABWD_Resumed','Policy_Period_LIHEAP_FY24_Cut','LIHEAP_Data_Available']

check = merged.groupby("Month_Year").agg(
    rows=("Zip Code", "size"),
    enroll_nonnull=("Enrollment_Count", "count"),
    **{f: (f, "nunique") for f in flags}
)

In [11]:
check

,rows,enroll_nonnull,LIHEAP_Policy_Active,LIHEAP_Expected_Direction,LIHEAP_Policy_Change,CalFresh_Policy_Change,FoodBank_Policy_Change,Policy_Period_EA_Ended,Policy_Period_ABWD_Resumed,Policy_Period_LIHEAP_FY24_Cut,LIHEAP_Data_Available
Month_Year,,,,,,,,,,,
2023-02,78,64,1,1,1,1,1,1,1,1,1
2023-03,95,77,1,1,1,1,1,1,1,1,1
2023-04,102,84,1,1,1,1,1,1,1,1,1
2023-05,103,80,1,1,1,1,1,1,1,1,1
2023-06,104,74,1,1,1,1,1,1,1,1,1
2023-07,107,74,1,1,1,1,1,1,1,1,1
2023-08,98,0,1,1,1,1,1,1,1,1,1
2023-09,89,0,1,1,1,1,1,1,1,1,1
2023-10,76,0,1,1,1,1,1,1,1,1,1


In [19]:
merged.to_csv("TimeLag_PolicyAnnotated.csv", index=False)